# 02 LLM coding with a promptbook

Sends each answer with the codebook (`codebooks/*.md`, converted to a JSON schema by `MaxNLP.make_json_schema`) to the LLM and stores the code confidences per run. Codebooks for causes, consequences, responses and lessons.

Cells tagged `api` call the LLM (OpenAI or Nebula) and were **not executed** for this repository. `dummy_data/` holds 12 synthetic citizen answers (plus 4 expert rows), random coding runs and random manual codings with the original file names. All results shown are meaningless.

In [1]:
# Folders. The original notebook read and wrote files in the project folder:
# DATA_DIR = OUT_DIR = "."
DATA_DIR = "dummy_data"   # synthetic example data shipped with this repo
OUT_DIR = "output"        # written by the notebook (gitignored)
import os; os.makedirs(OUT_DIR, exist_ok=True)

# ---- API switch -------------------------------------------------------------
# Cells starting with %%api call an external API (OpenAI, Nebula, Qualtrics, Eurostat, TypeSafe).
# RUN_API = False: these cells are skipped, so "Run All" makes no API calls and changes nothing on a server.
# RUN_API = True:  they run normally (needs the API keys; see README).
RUN_API = False
from IPython.core.magic import register_cell_magic
@register_cell_magic
def api(line, cell):
    if RUN_API:
        get_ipython().run_cell(cell)
    else:
        print("API cell skipped (RUN_API = False)")

In [2]:
import pandas as pd
import MaxNLP
from pathlib import Path
from tqdm.auto import tqdm
import json
tqdm.pandas(desc="Applying")
#%load_ext autoreload
#%autoreload 2
    
from datetime import date
date_str = date.today().isoformat()

In [3]:
# Load the translated text sample (25 per country --> 250 samples)
df_t=pd.read_json(f"{DATA_DIR}/2026-05-02 new translated.json")

# load the full data
df=pd.read_json(f"{DATA_DIR}/clean_df.json")

# Combine responses to one full response
#df["full_response"] = df.apply(MaxNLP.make_one_text, axis=1)

In [4]:
# Read the codebooks as Markdown

codebook_causes = Path("codebooks/2026-06-09 causes codebook.md").read_text(encoding="utf-8")
#codebook_responses = Path("codebooks/2026-01-30 responses codebook.md").read_text(encoding="utf-8")
#codebook_consequences = Path("codebooks/2026-04-08b consequences codebook.md").read_text(encoding="utf-8")
#codebook_futures = Path("codebooks/2026-02-15 lessons codebook.md").read_text(encoding="utf-8")

# Convert Markdown to Json
codebook_causes_json=MaxNLP.markdown_to_json(codebook_causes)
codebook_causes_json=sorted(codebook_causes_json, key=lambda d: d['label'])

#codebook_responses_json = MaxNLP.markdown_to_json(codebook_responses)
c#odebook_consequences_json=MaxNLP.markdown_to_json(codebook_consequences)
#codebook_futures_json=MaxNLP.markdown_to_json(codebook_futures)


# Get the code list for creating the json scheme.
codes_causes= [i["label"] for i in codebook_causes_json]
#codes_responses = [i["label"] for i in codebook_responses_json]
#codes_consequences = [i["label"] for i in codebook_consequences_json]
#codes_futures = [i["label"] for i in codebook_futures_json]

print("WORDS: ", len(codebook_causes.split()))
print("CHARACTERS: ", len(codebook_causes))
print("CODES: ",len(codes_causes))
print("Words/Code: ", round(len(codebook_causes.split())/len(codes_causes)))

# Check if all codes are imported with criteria & examples
for i in codebook_causes_json:
    print(i["label"])
    print(len(i['description'].split()),len(i['inclusion_criteria']),len(i['exclusion_criteria']),len(i['included_examples']))

#[i for i in codebook_causes_json if i["label"]=="Abstract societal and ideological explanations"]

NameError: name 'c' is not defined

In [5]:
# Create the System prompt --> requires to create a new client when changing the codebook.

PROMPTBOOK_INSTRUCTIONS = """
You are a helpful qualitative research assistant with knowledge about the EU energy system. You code short citizen perceptions of the European energy crisis with a defined codebook.
For each label, the codebook includes a definition with inclusion/exclusion criteria and examples that are coded with this label. 
You carefully read the full citizen perception and apply the codebook: What does the citizen say?
You only respond in correct json format. Task: For EACH label, output a confidence score (0.0 = impossible, 1.0 = certain). 
Multi-label allowed. Return JSON only, matching the schema exactly (no extra keys). 
Do not add any labels that are not in the schema.
This is your codebook for the question "What caused the EU energy crisis?":\n
""".strip()

In [6]:
%%api
# Load the Nebula Client
from dotenv import load_dotenv
import openAI_key

client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
available_models = MaxNLP.get_nebula_models(client)

print(f"Models: {available_models}\n\n")

run_x=1

## Important: Context window is set to 2000 = roughly 2000 words; larger codebooks require to change this.

API cell skipped (RUN_API = False)


In [7]:
%%api
### THIS IS TO RUN UNFINISHED ROUNDS --> STARTING FROM BATCH NR 69 for example.

date_str='2026-06-11'
codebook_name="causes"
code_list=codes_causes
codebook=json.dumps(codebook_causes_json, indent=2, ensure_ascii=False) # create a string from the json codebook
text_column='QID8'

import pandas as pd
import jsonlines
from pathlib import Path

start_row = 69 * 50  # = 3550

tmp = df.reset_index(names="orig_index") ### df_t is the translated sample; df is the full sample. 
tmp["batch_row"] = range(len(tmp))

tmp = tmp[tmp["batch_row"] >= start_row]

run_x=f"{4}_full" # uncomment when only partial run.

# Send the API request to the LLM; format everything with my module.
# Write each result as new line to the temp  batch files
res = tmp.progress_apply(
    lambda r: MaxNLP.code_text(
        r[text_column], r.orig_index, r.batch_row,
        client=client,
        model="FAST.gpt-oss:120b",
        code_list=code_list,
        PROMPTBOOK_INSTRUCTIONS=f"RUN-ID: {date_str}_{run_x}. "+PROMPTBOOK_INSTRUCTIONS+codebook,
        temperature=0.0,
        force_json_object=False,
        run_x=run_x,
        codebook_name=codebook_name
    ),
    axis=1,
)

# Combine the batch files to one json file
ann_df = pd.DataFrame(x for f in Path("annotations_tmp").glob(f"{date_str} run_{run_x}_{codebook_name}_batch_*.jsonl") for x in jsonlines.open(f)).pipe(
    lambda d: pd.json_normalize(d["annotation"])
    .set_index(d["orig_index"]))

# Writes the full run to one json file
ann_df.reset_index().to_json(f"{OUT_DIR}/{date_str} {codebook_name}_run_{run_x}.json", orient="records")


API cell skipped (RUN_API = False)


In [8]:
%%api
# Run the LLM coding in a for-loop

codebook_name="causes"
code_list=codes_causes
codebook=json.dumps(codebook_causes_json, indent=2, ensure_ascii=False) # create a string from the json codebook
text_column='QID8'

import pandas as pd
import jsonlines
from pathlib import Path

for run_x in range(5):

    # create new client to avoid cache:
    client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
    
    tmp = df.reset_index(names="orig_index") ### df_t is the translated sample; df is the full sample. 
    run_x=f"{run_x}_full" # uncomment when only partial run.

    print(f"{date_str}-{run_x}")
    
    # Split the set in batches (to rerun not all of them when there is an error)
    tmp["batch_row"] = range(len(tmp))

    
    
    # Send the API request to the LLM; format everything with my module.
    # Write each result as new line to the temp  batch files
    res = tmp.progress_apply(
        lambda r: MaxNLP.code_text(
            r[text_column], r.orig_index, r.batch_row,
            client=client,
            model="FAST.gpt-oss:120b",
            code_list=code_list,
            PROMPTBOOK_INSTRUCTIONS=f"RUN-ID: {date_str}_{run_x}. "+PROMPTBOOK_INSTRUCTIONS+codebook,
            temperature=0.0,
            force_json_object=False,
            run_x=run_x,
            codebook_name=codebook_name
        ),
        axis=1,
    )

    # Combine the batch files to one json file
    ann_df = pd.DataFrame(x for f in Path("annotations_tmp").glob(f"{date_str} run_{run_x}_{codebook_name}_batch_*.jsonl") for x in jsonlines.open(f)).pipe(
        lambda d: pd.json_normalize(d["annotation"])
        .set_index(d["orig_index"]))
    
    # Writes the full run to one json file
    ann_df.reset_index().to_json(f"{OUT_DIR}/{date_str} {codebook_name}_run_{run_x}.json", orient="records")


API cell skipped (RUN_API = False)
